# Magic number training debug

Short run on the `magic_number` example — watches whether the pipeline learns a fixed digit.

See `src/inspect_rl/example/magic_number.py` for the task definition.

In [ ]:
import os

# Pin training to one GPU (vLLM owns GPU 0). HF Trainer would otherwise spread
# params across all visible devices, breaking TRL's single-source NCCL sync.
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
os.environ["WANDB_DISABLED"] = "true"
# accelerate → deepspeed probe needs nvcc; nothing on PATH by default here.
os.environ["CUDA_HOME"] = "/opt/nvidia/hpc_sdk/Linux_aarch64/24.11/cuda/12.6"

import torch, trl, inspect_ai, transformers
print(f"torch: {torch.__version__} | cuda: {torch.cuda.is_available()} | devices: {torch.cuda.device_count()}")
print(f"trl: {trl.__version__} | inspect-ai: {inspect_ai.__version__} | transformers: {transformers.__version__}")
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — {free//1024**3}GB free / {total//1024**3}GB")

In [ ]:
import httpx

resp = httpx.get("http://localhost:8000/health/", timeout=5)
print(f"vLLM health: {resp.json()}")

# Quick test: ask the base policy "what is the magic number?"
# We tokenize client-side to match how rollouts will talk to vLLM.
from transformers import AutoTokenizer
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)

prompt_ids = tok.apply_chat_template(
    [{"role": "user", "content": "What is 2+2? Answer with a single digit."}],
    add_generation_prompt=True, tokenize=True, return_dict=False,
)
resp = httpx.post("http://localhost:8000/generate/", json={
    "prompts": [prompt_ids], "n": 1, "max_tokens": 30,
    "temperature": 0.0, "logprobs": 0,
}, timeout=30)
out = resp.json()
print(f"Base model test: {tok.decode(out['completion_ids'][0], skip_special_tokens=True)[:120]!r}")

In [ ]:
from inspect_ai import eval as inspect_eval, task_with
from inspect_ai.dataset import MemoryDataset
from inspect_ai.model import GenerateConfig, get_model
import inspect_rl.trl_vllm_provider  # noqa: F401 - registers @modelapi("trl-vllm")
from inspect_rl.example.magic_number import get_task, MAGIC_NUMBER

task = get_task(n=4)  # 4 identical samples — no need for more just to eyeball
samples = list(task.dataset)[:4]
print(f"MAGIC_NUMBER = {MAGIC_NUMBER}")
print(f"scorers: {[s.__qualname__.split('.')[0] for s in task.scorer]}")
print(f"sample[0].input[0].text = {samples[0].input[0].text!r}")
print(f"sample[0].target = {samples[0].target!r}")

In [ ]:
# Run the magic_number task once against the base policy via our provider.
# This exercises: react agent loop, submit tool, AgentAttempts retry,
# scorers, and the answer_only=True fix.
model = get_model(
    f"trl-vllm/{MODEL}",
    base_url="http://localhost:8000",
    tokenizer=tok,
    config=GenerateConfig(max_tokens=256, temperature=1.0),
    batch_timeout=1.0,
    max_batch_size=4,
)

logs = inspect_eval(
    task_with(task, dataset=MemoryDataset(samples=samples), epochs=1),
    model=model,
    max_samples=4,
    display="plain",
    fail_on_error=False,
)
log = logs[0]
print(f"status: {log.status}")
for s in (log.samples or []):
    scores = {n: sc.value for n, sc in (s.scores or {}).items()}
    print(f"  sample {s.id}: output={s.output.completion!r:30}  scores={scores}")

In [ ]:
# Dig into one sample to see why valid_submit scored 0 while magic_correctness scored 1.
s = log.samples[0]
print(f"sample id: {s.id}, output.completion = {s.output.completion!r}\n")
for i, m in enumerate(s.messages):
    tool_calls = getattr(m, "tool_calls", None)
    text_preview = (m.text or "")[:120].replace("\n", "\\n") if hasattr(m, "text") else ""
    print(f"  [{i}] {m.role:10} tool_calls={bool(tool_calls)}  text={text_preview!r}")
    if tool_calls:
        for tc in tool_calls:
            print(f"        → {tc.function}({tc.arguments}) parse_error={tc.parse_error!r}")

In [ ]:
# Re-load the task after the fix, re-run eval, re-inspect one sample.
import importlib, inspect_rl.example.magic_number as mn
importlib.reload(mn)
task2 = mn.get_task(n=4)
samples2 = list(task2.dataset)[:4]

logs2 = inspect_eval(
    task_with(task2, dataset=MemoryDataset(samples=samples2), epochs=1),
    model=model,
    max_samples=4,
    display="plain",
    fail_on_error=False,
)
log2 = logs2[0]
for s in (log2.samples or []):
    scores = {n: sc.value for n, sc in (s.scores or {}).items()}
    print(f"  sample {s.id}: output={s.output.completion!r:10}  scores={scores}")

print("\n--- messages from sample 0 ---")
s0 = log2.samples[0]
for i, m in enumerate(s0.messages):
    tool_calls = getattr(m, "tool_calls", None)
    text = (m.text or "")[:60].replace("\n", "\\n") if hasattr(m, "text") else ""
    print(f"  [{i}] {m.role:10} tool_calls={bool(tool_calls)}  text={text!r}")
    if tool_calls:
        for tc in tool_calls:
            print(f"        → {tc.function}({tc.arguments}) parse_error={tc.parse_error!r}")

In [ ]:
# Close any stale NCCL weight-update group on the vLLM server so the trainer
# can open a fresh one. No-op if there isn't one.
import httpx
try:
    httpx.post("http://localhost:8000/close_communicator/", json={}, timeout=5)
except Exception as e:
    print(f"close_communicator: {e}")
print("NCCL reset ok")

In [ ]:
# Short training run: 15 steps, batch/group of 8 (avoid OOM), no wandb.
# We reuse the reloaded mn module so the keep_in_messages=True fix is in play.
from trl import GRPOConfig
from inspect_rl.trainer import inspect_rl_train

task_train = mn.get_task(n=128)

grpo_config = GRPOConfig(
    output_dir="outputs/magic_number_debug",
    max_steps=15,
    per_device_train_batch_size=8,
    num_generations=8,
    max_completion_length=256,
    temperature=1.0,
    learning_rate=5e-6,
    warmup_steps=5,
    bf16=True,
    fp16=False,
    report_to="none",
    save_steps=100,          # don't bother saving in this smoke test
    logging_steps=1,
    use_vllm=True,
    vllm_mode="server",
    vllm_server_host="localhost",
    vllm_server_port=8000,
    reward_weights=[2.5, 0.1, -0.3],  # [magic_correctness, valid_submit, tool_call_failures]
)

print(f"Training {MODEL} for {grpo_config.max_steps} steps…")
trainer = inspect_rl_train(
    task=task_train,
    model=MODEL,
    grpo_config=grpo_config,
    vllm_base_url="http://localhost:8000",
    dataset_limit=128,
)
print("Training complete.")

In [ ]:
# Summarize the learning curve from the eval logs TRL wrote per rollout step.
from pathlib import Path
from inspect_ai.log import read_eval_log

run_dir = Path("outputs/magic_number_debug/2026-04-22-09-44-11")
log_files = sorted((run_dir / "eval_logs").glob("*.eval"))
print(f"{'step':>4} | {'correct':>7} | {'valid_sub':>9} | {'failures':>8} | {'tokens_out':>10}")
print("-" * 55)
for i, p in enumerate(log_files, 1):
    lg = read_eval_log(str(p))
    metrics = {es.name: es.metrics["accuracy"].value for es in lg.results.scores}
    out_toks = lg.stats.model_usage[list(lg.stats.model_usage.keys())[0]].output_tokens if lg.stats.model_usage else 0
    print(f"{i:>4} | {metrics.get('magic_correctness',0):>7.2f} | "
          f"{metrics.get('valid_submit',0):>9.2f} | "
          f"{metrics.get('tool_call_failures',0):>8.2f} | {out_toks:>10}")